## Part 1: Indexing Pipeline:
### Step 1: Data Loading:

In [1]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loader = PyPDFLoader("./data/data.pdf")

# Load all pages as Document objects
docs = loader.load()

print(f"Number of pages: {len(docs)}")
# print(docs[0].page_content)

Number of pages: 63


### Step 2: Data Splitting / Chunking

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
# print(chunks[0].page_content)
# print(chunks[0].metadata)

Number of chunks: 171


### Step 3: Data Conversion (Embeddings) and Vector Storage

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Create FAISS vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save the vector database
vector_store.save_local("./vector_store")

print(f"Indexed {len(chunks)} chunks.")
print("Vector database saved to ./vector_store")

Indexed 171 chunks.
Vector database saved to ./vector_store


## Part 2: Generation Pipeline
### Step 4: Retrieval:

In [4]:
retriever = vector_store.as_retriever(search_kwargs={"k":5})

import pandas as pd
from tqdm import tqdm

from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

from rouge_score import rouge_scorer



In [5]:
queries = pd.read_csv("./data/queries.csv")
# queries.head()

In [6]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
def retrieve_context(query):

    docs = retriever.invoke(query)

    retrieved_text = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return docs, retrieved_text


In [7]:
retrieval_results = []

for _, row in tqdm(queries.iterrows(), total=len(queries)):

    query = row["query"]
    golden_context = row["golden_context"]

    docs, retrieved_context = retrieve_context(query)

    ########################################################
    # Cross Encoder Similarity
    ########################################################

    ce_score = cross_encoder.predict(
        [(query, retrieved_context)]
    )[0]

    ########################################################
    # Binary Relevance
    ########################################################

    binary_hits = []

    for d in docs:

        score = cross_encoder.predict([(query, d.page_content)])[0]

        

        binary_hits.append(score > 2.0)

    retrieval_results.append({

        "query": query,
        "query_type": row["query_type"],

        "retrieved_context": retrieved_context,

        "CrossEncoderScore": ce_score,

        "BinaryHits": binary_hits

    })

100%|██████████| 20/20 [00:03<00:00,  6.24it/s]


In [8]:
def precision_at_k(binary_hits):

    return sum(binary_hits)/len(binary_hits)


def recall_at_k(binary_hits):

    if sum(binary_hits)==0:
        return 0

    return 1


def hit_rate(binary_hits):

    return int(any(binary_hits))


def reciprocal_rank(binary_hits):

    for i,x in enumerate(binary_hits):

        if x:
            return 1/(i+1)

    return 0


def average_precision(binary_hits):

    precisions=[]

    relevant=0

    for i,hit in enumerate(binary_hits):

        if hit:

            relevant+=1

            precisions.append(
                relevant/(i+1)
            )

    if len(precisions)==0:
        return 0

    return sum(precisions)/len(precisions)

In [9]:
metrics=[]

for r in retrieval_results:

    hits=r["BinaryHits"]

    metrics.append({

        "query":r["query"],

        "query_type":r["query_type"],

        "Precision@5":precision_at_k(hits),

        "Recall@5":recall_at_k(hits),

        "HitRate":hit_rate(hits),

        "MRR":reciprocal_rank(hits),

        "MAP":average_precision(hits),

        "CrossEncoderScore":r["CrossEncoderScore"]

    })

metrics_df=pd.DataFrame(metrics)

# metrics_df.head()

In [10]:
overall = metrics_df.drop(
    columns=["query","query_type"]
).mean()

print("Overall scores:")
print(overall)


Overall scores:
Precision@5          0.280000
Recall@5             0.800000
HitRate              0.800000
MRR                  0.750000
MAP                  0.750000
CrossEncoderScore    3.304783
dtype: float64


In [11]:
type_results = (

    metrics_df
    .groupby("query_type")
    .mean(numeric_only=True)

)

print(type_results)

               Precision@5  Recall@5  HitRate  MRR  MAP  CrossEncoderScore
query_type                                                                
aggregation           0.12       0.6      0.6  0.6  0.6           2.604614
comparison            0.36       0.8      0.8  0.8  0.8           4.291112
factoid               0.36       0.8      0.8  0.8  0.8           3.369186
summarization         0.28       1.0      1.0  0.8  0.8           2.954220


### Step 5: Augmentation:

In [9]:
prompt = f"""
Answer the question using only the retrieved documents.

Retrieved Documents:
{context}

Question:
{query}

Answer:
"""

### Step 6: Generation:

In [10]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = client.chat.completions.create(
    model="qwen3.5:2b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.0  # Greedy decoding
)

print(response.choices[0].message.content)

According to Document 1, hybrid search is a technique used to mitigate the limitations of vector database searches (aka semantic search technique) by combining traditional text search results with the text chunks linked to the retrieved vectors from the vector search. This combined hybrid text is then fed into a language model for generation.
